# Введение

Текущая задача - спрогнозировать стоимость аренды квартиры на основе её описания, расширив признаковое пространство, применив регуляризацию, масштабирование признаков, а также рассмотрев другие способы улучшения качества обучения моделей. Данные взяты с сайта объявлений о продаже квартир [renthop.com](https://www.renthop.com/).

# 0. Импорты

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, clone, TransformerMixin
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.model_selection import GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 1. Загрузка данных

Загрузим тот же датасет, который мы использовали в прошлом задании, и почистим выбросы по цене аренды квартир.

In [2]:
train_df = pd.read_json("../data/train.json")
train_df.head()

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        49352 non-null  float64
 1   bedrooms         49352 non-null  int64  
 2   building_id      49352 non-null  object 
 3   created          49352 non-null  object 
 4   description      49352 non-null  object 
 5   display_address  49352 non-null  object 
 6   features         49352 non-null  object 
 7   latitude         49352 non-null  float64
 8   listing_id       49352 non-null  int64  
 9   longitude        49352 non-null  float64
 10  manager_id       49352 non-null  object 
 11  photos           49352 non-null  object 
 12  price            49352 non-null  int64  
 13  street_address   49352 non-null  object 
 14  interest_level   49352 non-null  object 
dtypes: float64(3), int64(3), object(9)
memory usage: 6.0+ MB


In [4]:
df = train_df.copy()

low_limit = df["price"].quantile(0.01)
up_limit = df["price"].quantile(0.99)

df = df[(df["price"] > low_limit) & (df["price"] < up_limit)].copy()
df.reset_index(drop=True, inplace=True)
df.shape

(48343, 15)

# 2. Извлечение дополнительных признаков из столбца features

Чтобы увеличить качество прогноза моделей, к исходно используемым признакам *bedrooms* и *bathrooms* попробуем добавить дополнительную информацию. Такую информацию можно извлечь из столбца *features*. Взглянем на пример значения этого столбца.

In [5]:
df.loc[0, "features"]

['Dining Room',
 'Pre-War',
 'Laundry in Building',
 'Dishwasher',
 'Hardwood Floors',
 'Dogs Allowed',
 'Cats Allowed']

Здесь перечисляются возможности или особенности, которые арендатор получает вместе с квартирой: посудомоечная машинка, прачечная в здании, довоенная постройка и т.д. Эти характеристики также могут оказывать влияние на цену аренды. Отберём самые часто встречающиеся из них.

Данные в столбце уже представлены в виде списка строк, поэтому остаётся только обработать пробелы внутри самих характеристик.

In [6]:
df["features"] = df["features"].apply(
    lambda value: [feature.replace(" ", "") for feature in value] if isinstance(value, list) else value
)

df["features"].head()

0    [DiningRoom, Pre-War, LaundryinBuilding, Dishw...
1    [Doorman, Elevator, LaundryinBuilding, Dishwas...
2    [Doorman, Elevator, LaundryinBuilding, Laundry...
3                                                   []
4    [Doorman, Elevator, FitnessCenter, LaundryinBu...
Name: features, dtype: object

In [7]:
all_features = []
for index, row in df.iterrows():
    object_features = row["features"]
    all_features.extend(object_features)

len(set(all_features))

1531

In [8]:
ctr = Counter(all_features)
top_20 = ctr.most_common(20)
top_20

[('Elevator', 25375),
 ('HardwoodFloors', 23146),
 ('CatsAllowed', 23135),
 ('DogsAllowed', 21652),
 ('Doorman', 20479),
 ('Dishwasher', 20081),
 ('NoFee', 17793),
 ('LaundryinBuilding', 16082),
 ('FitnessCenter', 12989),
 ('Pre-War', 8971),
 ('LaundryinUnit', 8437),
 ('RoofDeck', 6417),
 ('OutdoorSpace', 5132),
 ('DiningRoom', 4890),
 ('HighSpeedInternet', 4223),
 ('Balcony', 2898),
 ('SwimmingPool', 2643),
 ('LaundryInBuilding', 2564),
 ('NewConstruction', 2504),
 ('Terrace', 2177)]

In [9]:
top_20_names = [feature for feature, count in top_20]

for feature in top_20_names:
    df[feature] = df["features"].apply(
        lambda value: 1 if feature in value else 0
    )

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48343 entries, 0 to 48342
Data columns (total 35 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   bathrooms          48343 non-null  float64
 1   bedrooms           48343 non-null  int64  
 2   building_id        48343 non-null  object 
 3   created            48343 non-null  object 
 4   description        48343 non-null  object 
 5   display_address    48343 non-null  object 
 6   features           48343 non-null  object 
 7   latitude           48343 non-null  float64
 8   listing_id         48343 non-null  int64  
 9   longitude          48343 non-null  float64
 10  manager_id         48343 non-null  object 
 11  photos             48343 non-null  object 
 12  price              48343 non-null  int64  
 13  street_address     48343 non-null  object 
 14  interest_level     48343 non-null  object 
 15  Elevator           48343 non-null  int64  
 16  HardwoodFloors     483

Также сохраним отдельно признаки, на которых будут обучаться модели.

In [10]:
feature_list = ["bathrooms", "bedrooms"] + top_20_names
len(feature_list)

22

# 3. Обучение моделей

## 3.1. Таблица результатов

Создадим три пустых датафрейма для сравнения метрик MAE, RMSE и R2 моделей на обучающей и тестовой выборках.

In [11]:
result_MAE = pd.DataFrame(columns=["model", "train", "test"])
result_RMSE = pd.DataFrame(columns=["model", "train", "test"])
result_R2 = pd.DataFrame(columns=["model", "train", "test"])

Также разделим данные на обучающую и тестовую выборки.

In [12]:
X = df[feature_list]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

Теперь пропишем программную реализацию коэффициента детерминации, исходя из формулы:
$$R^2 = 1 - \frac{RSS}{TSS},$$
где $RSS = \sum_{i=1}^{n} (y_{i true} - y_{i pred})^2$,

$TSS = \sum_{i=1}^{n} (y_{i true} - <y_{i true}>)^2$.

In [13]:
def r2_metric(y_true, y_pred):
    rss = ((y_true - y_pred)**2).sum()
    tss = ((y_true - y_true.mean())**2).sum()
    return (1 - rss / tss)

Дополнительно реализуем функцию, которая принимает в себя класс модели, её название и данные, после чего обучает модель, подсчитывает нужные метрики и добавляет результаты в датафреймы с метриками.

In [14]:
def model_metrics(model, model_name, X_train, X_test, y_train, y_test):
    local_model = clone(model)

    local_model.fit(X_train, y_train)
    pred_train = local_model.predict(X_train)
    pred_test = local_model.predict(X_test)
    
    train_mae = mean_absolute_error(y_train, pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    result_MAE.loc[len(result_MAE)] = [f"{model_name}", train_mae, test_mae]
    
    train_rmse = root_mean_squared_error(y_train, pred_train)
    test_rmse = root_mean_squared_error(y_test, pred_test)
    result_RMSE.loc[len(result_RMSE)] = [f"{model_name}", train_rmse, test_rmse]

    train_r2 = r2_metric(y_train, pred_train)
    test_r2 = r2_metric(y_test, pred_test)
    result_R2.loc[len(result_R2)] = [f"{model_name}", train_r2, test_r2]

## 3.2. Линейная регрессия

### 1) Собственная реализация

Задача линейной регрессии может сводиться к минимизации наименьших квадратов (МНК):
$$L(θ) = ||y - Xθ||^2 → min_θ$$ 
$$||y - Xθ||^2 = (y - Xθ)^T(y - Xθ) = (y^T - θ^TX^T)(y - Xθ) = y^Ty - y^TXθ - θ^TX^Ty + θ^TX^TXθ$$
$$y^TXθ = θ^TX^Ty - скаляры.$$
$$Следовательно, y^Ty - y^TXθ - θ^TX^Ty + θ^TX^TXθ = y^Ty - 2θ^TX^Ty + θ^TX^TXθ$$
Для решения этой задачи достаточно приравнять градиент функции потерь $L(θ)$ к нулю:
$$∇_θL = 2X^TXθ-2X^Ty$$
$$2X^TXθ-2X^Ty = 0$$
$$X^TXθ = X^Ty$$
$$θ = (X^TX)^{-1}X^Ty$$

In [15]:
class MyLinearRegression(BaseEstimator):
    def __init__(self):
        self.w = None

    def fit(self, X, y):
        X = np.column_stack([np.ones(X.shape[0]), X])
        self.w = np.linalg.inv(X.T @ X) @ X.T @ y
        return self

    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X @ self.w

In [16]:
my_lr = MyLinearRegression()

model_metrics(my_lr, "My linear regression", X_train, X_test, y_train, y_test)

### 2) Импорт из sklearn

In [17]:
lr = LinearRegression()

model_metrics(lr, "Linear regression (sklearn)", X_train, X_test, y_train, y_test)

In [18]:
result_MAE

,model,train,test
0,My linear regression,708.494998,709.678526
1,Linear regression (sklearn),708.494998,709.678526


In [19]:
result_RMSE

,model,train,test
0,My linear regression,1028.253868,1023.84289
1,Linear regression (sklearn),1028.253868,1023.84289


In [20]:
result_R2

,model,train,test
0,My linear regression,0.576715,0.593597
1,Linear regression (sklearn),0.576715,0.593597


По метрикам видно, что разницы между реализованной вручную линейной регрессией и моделью из библиотеки нет.

## 3.3. Регуляризованные модели (Ridge, Lasso, ElasticNet)

### 1) Собственная реализация

Для реализации L2-регуляризации воспользуемся аналитическим решением:
$$\theta = (X^TX + \alpha I)^{-1}X^Ty,$$
где $I$ - единичная матрица (появилась, потому что нужно было вынести $\theta$ за скобки).

Для остальных методов регуляризации нет аналитического решения, поэтому остаётся воспользоваться только градиентным спуском.

In [21]:
class MyRidge(BaseEstimator):
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.w = None

    def fit(self, X, y):
        X = np.column_stack([np.ones(X.shape[0]), X])
        n, d = X.shape
        I = np.eye(d)
        I[0, 0] = 0 
        self.w = np.linalg.inv(X.T @ X + self.alpha * I) @ X.T @ y
        return self

    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X @ self.w

In [22]:
class MyLasso(BaseEstimator):
    def __init__(self, alpha=1.0, max_iter=1000, lr = 0.01):
        self.alpha = alpha
        self.max_iter = max_iter
        self.lr = lr
        self.w = None

    def fit(self, X, y):
        X = np.column_stack([np.ones(X.shape[0]), X])
        self.w = np.zeros(X.shape[1])
        n = X.shape[0]

        for _ in range(self.max_iter):
            y_pred = X @ self.w
            error = y_pred - y
            grad = (2/n) * (X.T @ error) + self.alpha * np.sign(self.w)
            grad[0] = 0
            self.w = self.w - self.lr * grad
            
        return self

    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X @ self.w

In [23]:
class MyElasticNet(BaseEstimator):
    def __init__(self, alpha=1.0, max_iter=1000, l1_ratio=0.5, lr = 0.01):
        self.alpha = alpha
        self.max_iter = max_iter
        self.l1_ratio = l1_ratio
        self.lr = lr
        self.w = None

    def fit(self, X, y):
        X = np.column_stack([np.ones(X.shape[0]), X])
        self.w = np.zeros(X.shape[1])
        n = X.shape[0]

        for _ in range(self.max_iter):
            y_pred = X @ self.w
            error = y_pred - y
            grad = (2/n) * (X.T @ error) + self.l1_ratio * self.alpha * np.sign(self.w) + 2 * self.alpha * (1 - self.l1_ratio) * self.w
            grad[0] = 0
            self.w = self.w - self.lr * grad
            
        return self

    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X @ self.w

In [24]:
l2_reg = MyRidge()
model_metrics(l2_reg, "My Ridge (L2)", X_train, X_test, y_train, y_test)

l1_reg = MyLasso()
model_metrics(l1_reg, "My Lasso (L1)", X_train, X_test, y_train, y_test)

elnet_reg = MyElasticNet()
model_metrics(elnet_reg, "My ElasticNet", X_train, X_test, y_train, y_test)

### 2) Импорт из sklearn

In [25]:
l2_lib = Ridge()
model_metrics(l2_lib, "Ridge (sklearn)", X_train, X_test, y_train, y_test)

l1_lib = Lasso()
model_metrics(l1_lib, "Lasso (sklearn)", X_train, X_test, y_train, y_test)

elnet_lib = ElasticNet()
model_metrics(elnet_lib, "ElasticNet (sklearn)", X_train, X_test, y_train, y_test)

In [26]:
result_MAE

,model,train,test
0,My linear regression,708.494998,709.678526
1,Linear regression (sklearn),708.494998,709.678526
2,My Ridge (L2),708.491597,709.677352
3,My Lasso (L1),715.885081,718.667581
4,My ElasticNet,847.688474,857.979692
5,Ridge (sklearn),708.491597,709.677352
6,Lasso (sklearn),708.093245,709.569265
7,ElasticNet (sklearn),803.981791,810.987403


In [27]:
result_RMSE

,model,train,test
0,My linear regression,1028.253868,1023.842890
1,Linear regression (sklearn),1028.253868,1023.842890
2,My Ridge (L2),1028.253876,1023.846261
3,My Lasso (L1),1046.503402,1041.101215
4,My ElasticNet,1195.550974,1200.454469
5,Ridge (sklearn),1028.253876,1023.846261
6,Lasso (sklearn),1028.443194,1024.223384
7,ElasticNet (sklearn),1179.038398,1191.598341


In [28]:
result_R2

,model,train,test
0,My linear regression,0.576715,0.593597
1,Linear regression (sklearn),0.576715,0.593597
2,My Ridge (L2),0.576715,0.593594
3,My Lasso (L1),0.561556,0.579780
4,My ElasticNet,0.427772,0.441296
5,Ridge (sklearn),0.576715,0.593594
6,Lasso (sklearn),0.576559,0.593295
7,ElasticNet (sklearn),0.443470,0.449509


Имплементация методов регуляризации линейной регрессии по метрикам работает слегка хуже, чем из библиотеки.

Так как имплементация модели линейной регрессии с регуляризацией и без практически не отличается по метрикам от предложенных sklearn методов, то дальнейшие шаги будут осуществляться только над моделями из sklearn.

## 3.4. Нормализация (MinMax, StandardScaler)

Нормализация нужна в таких случаях:
- Для алгоритмов, основанных на градиентном спуске. Если признаки будут в разных масштабах, то функция потерь образует вытянутый элипсоид, что замедляет её сходимость.
- При регуляризации, иначе признаки с большим масштабом будут иметь маленький вес не из-за незначимости, а из-за единиц измерения.
- В целом, если признаки измеряются в разных шкалах. После нормализации веса станут сопоставимыми между собой.

Во многих моделях машинного обучения масштаб признаков влияет на прогноз: чем больше разброс значений, тем сильнее он влияет на прогноз.

Нормализация не нужна, если: 
- У признаков одинаковые измерительные шкалы или одинаковый масштаб.
- Алгоритмы - деревья решений и их производные, а также для наивного байесовского классификатора.

Так как нормализация нужна только для числовых признаков, то отдельно сохраним названия таких колонок. Остальные признаки у нас уже есть в переменной *top_20_names*.

In [29]:
base_columns = ["bathrooms", "bedrooms"]

### 1) Собственная реализация

Один из способов нормализации признаков - MinMaxScaler. Он масштабирует признаки до фиксированного диапазона (обычно \[0, 1\] или \[-1, 1\]):
$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

In [30]:
class MyMinMaxScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.min = None
        self.max = None

    def fit(self, X, y=None):
        self.min = X.min(axis=0)
        self.max = X.max(axis=0)
        return self

    def transform(self, X):
        return (X - self.min) / (self.max - self.min)

In [31]:
my_minmax = MyMinMaxScaler()
X_train_my_minmax = np.hstack([
    my_minmax.fit_transform(X_train[base_columns].values),
    X_train[top_20_names].values
])
X_test_my_minmax = np.hstack([
    my_minmax.transform(X_test[base_columns].values),
    X_test[top_20_names].values
])

In [32]:
pd.DataFrame(X_train_my_minmax[:5, :])

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.1,0.125,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.1,0.125,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.1,0.250,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.2,0.250,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.1,0.250,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Ещё один способ нормализации признаков - StandardScaler. Преобразует данные таким образом, что среднее значение становится равным 0, а стандартное отклонение - 1:
$$X_{scaled} = \frac{X -<X>}{\sigma}$$

In [33]:
class MyStandardScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mean = None
        self.sigma = None

    def fit(self, X, y=None):
        self.mean = X.mean(axis=0)
        self.sigma = X.std(axis=0)
        return self

    def transform(self, X):
        return (X - self.mean) / self.sigma

In [34]:
my_standard = MyStandardScaler()
X_train_my_standard = np.hstack([
    my_standard.fit_transform(X_train[base_columns].values),
    X_train[top_20_names].values
])
X_test_my_standard = np.hstack([
    my_standard.transform(X_test[base_columns].values),
    X_test[top_20_names].values
])

In [35]:
pd.DataFrame(X_train_my_standard[:5, :])

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,-0.427098,-0.486642,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.427098,-0.486642,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.427098,0.424361,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.774658,0.424361,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,-0.427098,0.424361,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 2) Импорт из sklearn

In [36]:
minmax = MinMaxScaler()
X_train_minmax = np.hstack([
    minmax.fit_transform(X_train[base_columns].values),
    X_train[top_20_names].values
])
X_test_minmax = np.hstack([
    minmax.transform(X_test[base_columns].values),
    X_test[top_20_names].values
])

In [37]:
pd.DataFrame(X_train_minmax[:5, :])

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.1,0.125,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.1,0.125,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.1,0.250,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.2,0.250,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.1,0.250,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
st_scaler = StandardScaler()
X_train_standard = np.hstack([
    st_scaler.fit_transform(X_train[base_columns].values),
    X_train[top_20_names].values
])
X_test_standard = np.hstack([
    st_scaler.transform(X_test[base_columns].values),
    X_test[top_20_names].values
])

In [39]:
pd.DataFrame(X_train_standard[:5, :])

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,-0.427098,-0.486642,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.427098,-0.486642,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.427098,0.424361,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.774658,0.424361,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,-0.427098,0.424361,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 3) Обучение моделей на нормализованных данных

In [40]:
# Обучение на собственной реализации MinMax
model_metrics(lr, "Linear regression (my MinMax)", X_train_my_minmax, X_test_my_minmax, y_train, y_test)
model_metrics(l2_lib, "Ridge (my MinMax)", X_train_my_minmax, X_test_my_minmax, y_train, y_test)
model_metrics(l1_lib, "Lasso (my MinMax)", X_train_my_minmax, X_test_my_minmax, y_train, y_test)
model_metrics(elnet_lib, "ElasticNet (my MinMax)", X_train_my_minmax, X_test_my_minmax, y_train, y_test)

In [41]:
# Обучение на MinMax из sklearn
model_metrics(lr, "Linear regression (sklearn MinMax)", X_train_minmax, X_test_minmax, y_train, y_test)
model_metrics(l2_lib, "Ridge (sklearn MinMax)", X_train_minmax, X_test_minmax, y_train, y_test)
model_metrics(l1_lib, "Lasso (sklearn MinMax)", X_train_minmax, X_test_minmax, y_train, y_test)
model_metrics(elnet_lib, "ElasticNet (sklearn MinMax)", X_train_minmax, X_test_minmax, y_train, y_test)

In [42]:
# Обучение на собственной реализации StandardScaler
model_metrics(lr, "Linear regression (my StandardScaler)", X_train_my_standard, X_test_my_standard, y_train, y_test)
model_metrics(l2_lib, "Ridge (my StandardScaler)", X_train_my_standard, X_test_my_standard, y_train, y_test)
model_metrics(l1_lib, "Lasso (my StandardScaler)", X_train_my_standard, X_test_my_standard, y_train, y_test)
model_metrics(elnet_lib, "ElasticNet (my StandardScaler)", X_train_my_standard, X_test_my_standard, y_train, y_test)

In [43]:
# Обучение на StandardScaler из sklearn
model_metrics(lr, "Linear regression (sklearn StandardScaler)", X_train_my_standard, X_test_my_standard, y_train, y_test)
model_metrics(l2_lib, "Ridge (sklearn StandardScaler)", X_train_my_standard, X_test_my_standard, y_train, y_test)
model_metrics(l1_lib, "Lasso (sklearn StandardScaler)", X_train_my_standard, X_test_my_standard, y_train, y_test)
model_metrics(elnet_lib, "ElasticNet (sklearn StandardScaler)", X_train_my_standard, X_test_my_standard, y_train, y_test)

In [44]:
result_MAE

,model,train,test
0,My linear regression,708.494998,709.678526
1,Linear regression (sklearn),708.494998,709.678526
2,My Ridge (L2),708.491597,709.677352
3,My Lasso (L1),715.885081,718.667581
4,My ElasticNet,847.688474,857.979692
5,Ridge (sklearn),708.491597,709.677352
6,Lasso (sklearn),708.093245,709.569265
7,ElasticNet (sklearn),803.981791,810.987403
8,Linear regression (my MinMax),708.494998,709.678526
9,Ridge (my MinMax),708.614152,709.884872


In [45]:
result_RMSE

,model,train,test
0,My linear regression,1028.253868,1023.842890
1,Linear regression (sklearn),1028.253868,1023.842890
2,My Ridge (L2),1028.253876,1023.846261
3,My Lasso (L1),1046.503402,1041.101215
4,My ElasticNet,1195.550974,1200.454469
5,Ridge (sklearn),1028.253876,1023.846261
6,Lasso (sklearn),1028.443194,1024.223384
7,ElasticNet (sklearn),1179.038398,1191.598341
8,Linear regression (my MinMax),1028.253868,1023.842890
9,Ridge (my MinMax),1028.303518,1024.070144


In [46]:
result_R2

,model,train,test
0,My linear regression,0.576715,0.593597
1,Linear regression (sklearn),0.576715,0.593597
2,My Ridge (L2),0.576715,0.593594
3,My Lasso (L1),0.561556,0.579780
4,My ElasticNet,0.427772,0.441296
5,Ridge (sklearn),0.576715,0.593594
6,Lasso (sklearn),0.576559,0.593295
7,ElasticNet (sklearn),0.443470,0.449509
8,Linear regression (my MinMax),0.576715,0.593597
9,Ridge (my MinMax),0.576674,0.593416


Результаты нормализации вручную написанных MinMaxScaler и StandardScaler не отличается по значениям от аналогичных методов из sklearn. Также не отличаются и метрики. Единственное, что при сочетании MinMaxScaler с ElasticNet у последней рухнула точность. Причина в том, что MinMaxScaler сжимает данные в наименьший масштаб - \[0, 1\]. При $\alpha=1.0$ штраф ElasticNet (L1+L2 одновременно) становится настолько большим относительно масштаба данных, что модель просто стягивает почти все веса к нулю, вызывая тем самым сильное недообучение.

## 3.5. Переобучение

Создадим упрощённый пример переобучения модели. Для этого возведём признаки в 10-ю степень.

In [47]:
poly = PolynomialFeatures(degree=10)

X_train_poly = np.hstack([
    poly.fit_transform(X_train[base_columns]),
    X_train[top_20_names].values
])

X_test_poly = np.hstack([
    poly.transform(X_test[base_columns]),
    X_test[top_20_names].values
])

In [48]:
model_metrics(lr, "Linear regression polynomial", X_train_poly, X_test_poly, y_train, y_test)
model_metrics(l2_lib, "Ridge polynomial", X_train_poly, X_test_poly, y_train, y_test)
model_metrics(l1_lib, "Lasso polynomial", X_train_poly, X_test_poly, y_train, y_test)
model_metrics(elnet_lib, "ElasticNet polynomial", X_train_poly, X_test_poly, y_train, y_test)

/mnt/d/School-21_tasks/ML2_Supervised_learning.ID_1254799-1/src/env/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.941e+10, tolerance: 9.660e+06
  model = cd_fast.enet_coordinate_descent(
/mnt/d/School-21_tasks/ML2_Supervised_learning.ID_1254799-1/src/env/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.191e+10, tolerance: 9.660e+06
  model = cd_fast.enet_coordinate_descent(


In [49]:
result_MAE

,model,train,test
0,My linear regression,708.494998,709.678526
1,Linear regression (sklearn),708.494998,709.678526
2,My Ridge (L2),708.491597,709.677352
3,My Lasso (L1),715.885081,718.667581
4,My ElasticNet,847.688474,857.979692
5,Ridge (sklearn),708.491597,709.677352
6,Lasso (sklearn),708.093245,709.569265
7,ElasticNet (sklearn),803.981791,810.987403
8,Linear regression (my MinMax),708.494998,709.678526
9,Ridge (my MinMax),708.614152,709.884872


In [50]:
result_RMSE

,model,train,test
0,My linear regression,1028.253868,1023.842890
1,Linear regression (sklearn),1028.253868,1023.842890
2,My Ridge (L2),1028.253876,1023.846261
3,My Lasso (L1),1046.503402,1041.101215
4,My ElasticNet,1195.550974,1200.454469
5,Ridge (sklearn),1028.253876,1023.846261
6,Lasso (sklearn),1028.443194,1024.223384
7,ElasticNet (sklearn),1179.038398,1191.598341
8,Linear regression (my MinMax),1028.253868,1023.842890
9,Ridge (my MinMax),1028.303518,1024.070144


In [51]:
result_R2

,model,train,test
0,My linear regression,0.576715,0.593597
1,Linear regression (sklearn),0.576715,0.593597
2,My Ridge (L2),0.576715,0.593594
3,My Lasso (L1),0.561556,0.579780
4,My ElasticNet,0.427772,0.441296
5,Ridge (sklearn),0.576715,0.593594
6,Lasso (sklearn),0.576559,0.593295
7,ElasticNet (sklearn),0.443470,0.449509
8,Linear regression (my MinMax),0.576715,0.593597
9,Ridge (my MinMax),0.576674,0.593416


На обычной линейной регрессии и регрессии с R2-регуляризацией применение полиномиальных признаков показало себя хуже всех - коэффициент детерминации даже ушёл в отрицательные значения. А вот на Lasso это дало обратный эффект - и модель стала лучшей по метрикам.

## 3.6 Подбор параметра альфа

$\alpha$ задаёт силу штрафа в моделях с регуляризацией. Некоторые модели после нормализации данных показали результаты хуже, чем без неё. Это случилось, потому что данные были масштабированы, но на веса признаков накаладывался тот же большой штраф, что и без масштабирования признаков.

In [52]:
param_grid = {"alpha": [1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]}

In [53]:
def find_best_alpha(model, grid, X_train, y_train):
    gs = GridSearchCV(model, grid, cv=5, scoring="r2")
    gs.fit(X_train, y_train)
    return gs.best_params_

In [54]:
print(f"Лучшее альфа для Ridge: {find_best_alpha(Ridge(), param_grid, X_train, y_train)}")
print(f"Лучшее альфа для Ridge MinMax: {find_best_alpha(Ridge(), param_grid, X_train_minmax, y_train)}")
print(f"Лучшее альфа для Ridge StandardScaler {find_best_alpha(Ridge(), param_grid, X_train_standard, y_train)}")

Лучшее альфа для Ridge: {'alpha': 10}
Лучшее альфа для Ridge MinMax: {'alpha': 0.1}
Лучшее альфа для Ridge StandardScaler {'alpha': 10}


In [55]:
print(f"Лучшее альфа для Lasso: {find_best_alpha(Lasso(), param_grid, X_train, y_train)}")
print(f"Лучшее альфа для Lasso MinMax: {find_best_alpha(Lasso(), param_grid, X_train_minmax, y_train)}")
print(f"Лучшее альфа для Lasso StandardScaler {find_best_alpha(Lasso(), param_grid, X_train_standard, y_train)}")

Лучшее альфа для Lasso: {'alpha': 0.01}
Лучшее альфа для Lasso MinMax: {'alpha': 0.01}
Лучшее альфа для Lasso StandardScaler {'alpha': 0.01}


In [56]:
print(f"Лучшее альфа для ElasticNet: {find_best_alpha(ElasticNet(), param_grid, X_train, y_train)}")
print(f"Лучшее альфа для ElasticNet MinMax: {find_best_alpha(ElasticNet(), param_grid, X_train_minmax, y_train)}")
print(f"Лучшее альфа для ElasticNet StandardScaler {find_best_alpha(ElasticNet(), param_grid, X_train_standard, y_train)}")

Лучшее альфа для ElasticNet: {'alpha': 0.001}
Лучшее альфа для ElasticNet MinMax: {'alpha': 1e-05}
Лучшее альфа для ElasticNet StandardScaler {'alpha': 0.001}


In [57]:
model_metrics(Ridge(alpha=10), "Ridge (best alpha)", X_train, X_test, y_train, y_test)
model_metrics(Ridge(alpha=0.1), "Ridge MinMax (best alpha)", X_train_minmax, X_test_minmax, y_train, y_test)
model_metrics(Ridge(alpha=10), "Ridge StandardScaler (best alpha)", X_train_standard, X_test_standard, y_train, y_test)

model_metrics(Lasso(alpha=0.01), "Lasso (best alpha)", X_train, X_test, y_train, y_test)
model_metrics(Lasso(alpha=0.01), "Lasso MinMax (best alpha)", X_train_minmax, X_test_minmax, y_train, y_test)
model_metrics(Lasso(alpha=0.01), "Lasso StandardScaler (best alpha)", X_train_standard, X_test_standard, y_train, y_test)

model_metrics(ElasticNet(alpha=0.001), "ElasticNet (best alpha)", X_train, X_test, y_train, y_test)
model_metrics(ElasticNet(alpha=1e-05), "ElasticNet MinMax (best alpha)", X_train_minmax, X_test_minmax, y_train, y_test)
model_metrics(ElasticNet(alpha=0.001), "ElasticNet StandardScaler (best alpha)", X_train_standard, X_test_standard, y_train, y_test)

## 3.7. Наивные модели (по среднему арифметическому и медиане)

Чтобы увидеть, насколько эффективны наши текущие модели, нужно сравнить их с самыми простыми решениями - наивными моделями.

In [58]:
naive_mean = DummyRegressor(strategy="mean")
model_metrics(naive_mean, "Naive mean", X_train, X_test, y_train, y_test)

In [59]:
naive_median = DummyRegressor(strategy="median")
model_metrics(naive_median, "Naive median", X_train, X_test, y_train, y_test)

# 4. Результаты

In [60]:
result_MAE

,model,train,test
0,My linear regression,708.494998,709.678526
1,Linear regression (sklearn),708.494998,709.678526
2,My Ridge (L2),708.491597,709.677352
3,My Lasso (L1),715.885081,718.667581
4,My ElasticNet,847.688474,857.979692
5,Ridge (sklearn),708.491597,709.677352
6,Lasso (sklearn),708.093245,709.569265
7,ElasticNet (sklearn),803.981791,810.987403
8,Linear regression (my MinMax),708.494998,709.678526
9,Ridge (my MinMax),708.614152,709.884872


In [ ]:
result_MAE

,model,train,test
0,My linear regression,708.494998,709.678526
1,Linear regression (sklearn),708.494998,709.678526
2,My Ridge (L2),708.491597,709.677352
3,My Lasso (L1),715.885081,718.667581
4,My ElasticNet,847.688474,857.979692
5,Ridge (sklearn),708.491597,709.677352
6,Lasso (sklearn),708.093245,709.569265
7,ElasticNet (sklearn),803.981791,810.987403
8,Linear regression (my MinMax),708.494998,709.678526
9,Ridge (my MinMax),708.614152,709.884872


In [61]:
result_RMSE

,model,train,test
0,My linear regression,1028.253868,1023.842890
1,Linear regression (sklearn),1028.253868,1023.842890
2,My Ridge (L2),1028.253876,1023.846261
3,My Lasso (L1),1046.503402,1041.101215
4,My ElasticNet,1195.550974,1200.454469
5,Ridge (sklearn),1028.253876,1023.846261
6,Lasso (sklearn),1028.443194,1024.223384
7,ElasticNet (sklearn),1179.038398,1191.598341
8,Linear regression (my MinMax),1028.253868,1023.842890
9,Ridge (my MinMax),1028.303518,1024.070144


In [62]:
result_R2

,model,train,test
0,My linear regression,0.576715,0.593597
1,Linear regression (sklearn),0.576715,0.593597
2,My Ridge (L2),0.576715,0.593594
3,My Lasso (L1),0.561556,0.579780
4,My ElasticNet,0.427772,0.441296
5,Ridge (sklearn),0.576715,0.593594
6,Lasso (sklearn),0.576559,0.593295
7,ElasticNet (sklearn),0.443470,0.449509
8,Linear regression (my MinMax),0.576715,0.593597
9,Ridge (my MinMax),0.576674,0.593416


По коэффициенту детерминации наилучшей моделью оказалась полиномиальная регрессия с L1-регуляризацией. Также она оказалась самой стабильной по метрикам MAE и RMSE из всех моделей.

Подбор параметра альфа способствовал улучшению результатов в отдельных случаях, но ни одна модель, кроме вышеуказанной, по коэффициенту детерминации не превзошла обычную линейную регрессию.

# 5. Дополнительно

## 5.1. Логарифмирование целевой переменной

В предыдущем задании на графике распределения целевой переменной можно было увидеть явно выраженное ассиметричное распределение с хвостом в правой стороне. Если у нас есть распределение с тяжёлыми хвостами, можно использовать монотонную функцию, чтобы "улучшить" распределение. Для данного случая подойдёт логарифм.

Поэтому мы логарифмируем целевую переменную в обучающей выборке, а для сопоставимости результатов трансформируем предсказание обратно через экспоненту.

In [63]:
# Вычисляем ln(1 + y), чтобы в случае, когда y = 0, не возникла ошибка
y_train_log = np.log1p(y_train)

In [64]:
print(f"Лучшее альфа для Ridge MinMax (log): {find_best_alpha(Ridge(), param_grid, X_train_minmax, y_train_log)}")

Лучшее альфа для Ridge MinMax (log): {'alpha': 0.1}


In [65]:
# Переделаем функцию model_metrics, чтобы она корректно обрабатывала логарифмированные данные
def model_metrics_log(model, model_name, X_train, X_test, y_train_log, y_test):
    local_model = clone(model)

    local_model.fit(X_train, y_train_log)
    pred_train_log = local_model.predict(X_train)
    pred_train = np.expm1(pred_train_log)
    pred_test_log = local_model.predict(X_test)
    pred_test = np.expm1(pred_test_log)
    
    train_mae = mean_absolute_error(np.expm1(y_train_log), pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    result_MAE.loc[len(result_MAE)] = [f"{model_name}", train_mae, test_mae]
    
    train_rmse = root_mean_squared_error(np.expm1(y_train_log), pred_train)
    test_rmse = root_mean_squared_error(y_test, pred_test)
    result_RMSE.loc[len(result_RMSE)] = [f"{model_name}", train_rmse, test_rmse]

    train_r2 = r2_metric(np.expm1(y_train_log), pred_train)
    test_r2 = r2_metric(y_test, pred_test)
    result_R2.loc[len(result_R2)] = [f"{model_name}", train_r2, test_r2]

In [66]:
model_metrics_log(Ridge(alpha=0.1), "Sklearn Ridge MinMax (log)", X_train_minmax, X_test_minmax, y_train_log, y_test)

In [67]:
result_MAE.iloc[-1:, :]

,model,train,test
39,Sklearn Ridge MinMax (log),694.839675,698.272811


In [68]:
result_RMSE.iloc[-1:, :]

,model,train,test
39,Sklearn Ridge MinMax (log),1053.411926,1034.419592


In [69]:
result_R2.iloc[-1:, :]

,model,train,test
39,Sklearn Ridge MinMax (log),0.555748,0.585157


У нас незначительно улучшились значения метрики MAE для обеих выборок, в то время как для остальных метрик значения ухудшились. По коэффициенту детерминации видно, что из-за более обобщённого вида целевой переменной модель теперь хуже показывает себя на обучающей выборке, но в то же время лучше улавливает закономерности в тестовой выборке. Это заметно по большему разрыву в значениях метрики R2 для обеих выборок.

## 5.2. Удаление выбросов только из обучающих данных

В процессе обучения моделям нужно выявить общую закономерность в данных, а не подстраиваться под все возможные значения. Поэтому возникает необходимость удаления выбросов в целевой переменной из обучающего набора данных. В то время как задача тестовой выборки - проверить работу модели в реальных условиях, где выбросы действительно могут встретиться. В связи с этим из неё выбросы удалять не нужно.

In [70]:
outliers_df = train_df.copy()

for feature in top_20_names:
    outliers_df[feature] = outliers_df["features"].apply(
        lambda value: 1 if feature in value else 0
    )

outliers_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 35 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   bathrooms          49352 non-null  float64
 1   bedrooms           49352 non-null  int64  
 2   building_id        49352 non-null  object 
 3   created            49352 non-null  object 
 4   description        49352 non-null  object 
 5   display_address    49352 non-null  object 
 6   features           49352 non-null  object 
 7   latitude           49352 non-null  float64
 8   listing_id         49352 non-null  int64  
 9   longitude          49352 non-null  float64
 10  manager_id         49352 non-null  object 
 11  photos             49352 non-null  object 
 12  price              49352 non-null  int64  
 13  street_address     49352 non-null  object 
 14  interest_level     49352 non-null  object 
 15  Elevator           49352 non-null  int64  
 16  HardwoodFloors     49352 n

In [71]:
train_data, test_data = train_test_split(outliers_df, test_size=0.22, random_state=21)
train_data.shape

(38494, 35)

In [72]:
train_data = train_data[(train_data["price"] > low_limit) & (train_data["price"] < up_limit)].copy()
train_data.reset_index(drop=True, inplace=True)
train_data.shape

(37691, 35)

In [73]:
X_train_o = train_data[feature_list]
y_train_o = train_data["price"]

X_test_o = test_data[feature_list]
y_test_o = test_data["price"]

In [74]:
model_metrics(lr, "Linear regression (outliers test)", X_train_o, X_test_o, y_train_o, y_test_o)
model_metrics(Ridge(alpha=0.1), "Ridge (outliers test)", X_train_o, X_test_o, y_train_o, y_test_o)
model_metrics(Lasso(alpha=0.01), "Lasso (outliers test)", X_train_o, X_test_o, y_train_o, y_test_o)

In [75]:
result_MAE.iloc[-3:, :]

,model,train,test
40,Linear regression (outliers test),732.095995,840.453846
41,Ridge (outliers test),732.095841,840.453882
42,Lasso (outliers test),732.093451,840.449166


In [76]:
result_RMSE.iloc[-3:, :]

,model,train,test
40,Linear regression (outliers test),1054.740311,2121.341415
41,Ridge (outliers test),1054.740311,2121.343430
42,Lasso (outliers test),1054.740316,2121.345069


In [77]:
result_R2.iloc[-3:, :]

,model,train,test
40,Linear regression (outliers test),0.558558,0.349966
41,Ridge (outliers test),0.558558,0.349965
42,Lasso (outliers test),0.558558,0.349964


При удалении выбросов только из обучающей выборки значения метрик для моделей на тестовой выборке значительно ухудшились. Это говорит о том, что модель плохо справляется с менее предсказуемыми данными - и ей не хватает данных из обучающей выборки дял усвоения закономерностей для прогнозирования.

## 5.3. Реализация линейной регрессии с пакетным и мини-пакетным обучением

In [78]:
class MyBatchRegression(BaseEstimator):
    def __init__(self, learning_rate=0.01, epochs=100):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.w = None

    def fit(self, X, y):
        X = np.column_stack([np.ones(X.shape[0]), X])
        samples, features = X.shape
        self.w = np.zeros(features)
        
        for epoch in range(self.epochs):
            y_pred = X @ self.w
            error = y_pred - y
            gradient = (2 / samples) * (X.T @ error)
            self.w -= self.learning_rate * gradient
        
        return self

    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X @ self.w

In [79]:
my_br = MyBatchRegression()

model_metrics(my_br, "My batch linear regression", X_train, X_test, y_train, y_test)

In [80]:
class MyMiniBatchRegression(BaseEstimator):
    def __init__(self, learning_rate=0.01, epochs=100, batch_size=64):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.w = None

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        
        X = np.column_stack([np.ones(X.shape[0]), X])
        samples, features = X.shape
        self.w = np.zeros(features)
        
        for epoch in range(self.epochs):
            ids = np.random.permutation(samples)
            X_shuffled = X[ids]
            y_shuffled = y[ids]

            for i in range(0, samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]
                
                y_pred_batch = X_batch @ self.w
                error_batch = y_pred_batch - y_batch
                gradient = (2 / len(X_batch)) * (X_batch.T @ error_batch)
                self.w -= self.learning_rate * gradient
        
        return self

    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X @ self.w

In [81]:
my_mbr = MyMiniBatchRegression()

model_metrics(my_mbr, "My mini-batch linear regression", X_train, X_test, y_train, y_test)

In [82]:
result_MAE.iloc[-2:, :]

,model,train,test
43,My batch linear regression,772.444452,783.216225
44,My mini-batch linear regression,711.939950,713.212895


In [83]:
result_RMSE.iloc[-2:, :]

,model,train,test
43,My batch linear regression,1110.473525,1118.184217
44,My mini-batch linear regression,1029.489187,1025.547736


In [84]:
result_R2.iloc[-2:, :]

,model,train,test
43,My batch linear regression,0.506316,0.515251
44,My mini-batch linear regression,0.575697,0.592242


Линейная регрессия с пакетным градиентным спуском показала себя хуже обычной линейной регрессии, в то время как с мини-пакетным градиентным спуском - чуть лучше на обучающей, но немного хуже на тестовой выборках.